# Rulebook RAG Exploration

This notebook is for exploring and debugging the rulebook RAG pipeline.

In [ ]:
import sys
sys.path.insert(0, '..')

from pathlib import Path
from dotenv import load_dotenv

load_dotenv()

## Load a Parsed Rulebook

In [ ]:
from src.ingestion.parse_loader import ParseLoader

# Update this path to your parsed rulebook JSON
rulebook_path = Path('../data/parsed/example.json')

if rulebook_path.exists():
    loader = ParseLoader(rulebook_path)
    document, blocks = loader.load()
    print(f"Loaded: {document.name}")
    print(f"Pages: {document.total_pages}")
    print(f"Blocks: {len(blocks)}")
else:
    print(f"File not found: {rulebook_path}")
    print("Place a parsed rulebook JSON in data/parsed/")

## Explore Block Types

In [ ]:
from collections import Counter

if 'blocks' in dir():
    type_counts = Counter(b.block_type.value for b in blocks)
    for block_type, count in type_counts.most_common():
        print(f"{block_type}: {count}")

## Mini-Page Detection

In [ ]:
from src.ingestion.mini_page_detector import MiniPageDetector

if 'blocks' in dir():
    detector = MiniPageDetector()
    
    # Get blocks for page 0
    page_0_blocks = [b for b in blocks if b.pdf_page == 0]
    
    if page_0_blocks:
        # Estimate page dimensions
        max_x = max(b.bbox.x1 for b in page_0_blocks)
        max_y = max(b.bbox.y1 for b in page_0_blocks)
        
        layout = detector.detect_layout(page_0_blocks, max_x * 1.1, max_y * 1.1)
        print(f"Columns: {layout.num_columns}")
        print(f"Rows: {layout.num_rows}")
        print(f"Mini-pages: {layout.mini_page_count}")

## Build Document Graph

In [ ]:
from src.graph.graph_builder import GraphBuilder

if 'blocks' in dir() and 'document' in dir():
    builder = GraphBuilder(document, blocks)
    graph = builder.build()
    
    print(f"Nodes: {len(graph.blocks)}")
    print(f"Edges: {len(graph.edges)}")
    
    edge_types = Counter(e.edge_type.value for e in graph.edges)
    for edge_type, count in edge_types.most_common():
        print(f"  {edge_type}: {count}")

## Create Chunks

In [ ]:
from src.chunking.chunk_builder import ChunkBuilder

if 'graph' in dir():
    chunk_builder = ChunkBuilder(max_tokens=512, min_tokens=50)
    chunks = chunk_builder.build_chunks(graph)
    chunks = chunk_builder.merge_small_chunks(chunks)
    
    print(f"Created {len(chunks)} chunks")
    
    for i, chunk in enumerate(chunks[:3]):
        print(f"\n--- Chunk {i+1} ---")
        print(f"Tokens: {chunk.token_count}")
        print(f"Page: {chunk.pdf_page}")
        print(f"Text preview: {chunk.text[:200]}...")

## Generate Embeddings

In [ ]:
from src.embeddings.embedder import Embedder
import os

if 'chunks' in dir() and os.getenv('OPENAI_API_KEY'):
    embedder = Embedder()
    
    # Embed just the first few chunks for testing
    test_chunks = chunks[:3]
    test_chunks = embedder.embed_chunks(test_chunks)
    
    print(f"Embedded {len(test_chunks)} chunks")
    print(f"Embedding dimension: {len(test_chunks[0].embedding)}")
else:
    print("Set OPENAI_API_KEY to generate embeddings")

## Visualize Block Layout

Optional: requires matplotlib

In [ ]:
try:
    import matplotlib.pyplot as plt
    import matplotlib.patches as patches
    
    if 'blocks' in dir():
        # Visualize page 0
        page_blocks = [b for b in blocks if b.pdf_page == 0]
        
        fig, ax = plt.subplots(figsize=(10, 12))
        
        colors = {
            'text': 'lightblue',
            'title': 'lightgreen',
            'figure': 'lightyellow',
            'table': 'lightcoral',
        }
        
        for block in page_blocks:
            color = colors.get(block.block_type.value, 'lightgray')
            rect = patches.Rectangle(
                (block.bbox.x0, block.bbox.y0),
                block.bbox.width,
                block.bbox.height,
                linewidth=1,
                edgecolor='black',
                facecolor=color,
                alpha=0.5
            )
            ax.add_patch(rect)
        
        ax.set_xlim(0, max(b.bbox.x1 for b in page_blocks) * 1.1)
        ax.set_ylim(max(b.bbox.y1 for b in page_blocks) * 1.1, 0)  # Invert Y axis
        ax.set_aspect('equal')
        ax.set_title('Page 0 Block Layout')
        plt.show()
except ImportError:
    print("matplotlib not installed, skipping visualization")